# Customizing agent memory

By default, agents use AgentState to manage short term memory, specifically the conversation history via a messages key.
You can extend AgentState to add additional fields. Custom state schemas are passed to create_agent using the state_schema parameter.

In [1]:
import os

from dotenv import load_dotenv

load_dotenv()

True

In [4]:
import pprint
import pprint
from langchain.tools import tool

from langchain.chat_models import init_chat_model
from rich import print as rprint
from langchain.agents import create_agent, AgentState
from langgraph.checkpoint.memory import InMemorySaver


In [3]:
model_free = init_chat_model("openai/gpt-oss-20b",
                        api_key=os.environ["GROQ_API_KEY"],
                        model_provider="groq",
                        # base_url="https://api.groq.com/openai/v1",
                        max_tokens=10000, temperature=0.0)

model_basic = init_chat_model("openrouter/free",
                        api_key=os.environ["OPENROUTER_API_KEY"],
                        model_provider="openrouter",
                        base_url="https://openrouter.ai/api/v1",
                        max_tokens=1000, temperature=0.0)

model_medium = init_chat_model("openai/gpt-5.6-luna",
                        api_key=os.environ["OPENROUTER_API_KEY"],
                        model_provider="openrouter",
                        base_url="https://openrouter.ai/api/v1",
                        max_tokens=10000, temperature=0.0)

model_advanced = init_chat_model("openai/gpt-5.6-luna-pro",
                        api_key=os.environ["OPENROUTER_API_KEY"],
                        model_provider="openrouter",
                        base_url="https://openrouter.ai/api/v1",
                        max_tokens=10000, temperature=0.0)

model_safety = init_chat_model("nvidia/nemotron-3.5-content-safety:free",
                        api_key=os.environ["OPENROUTER_API_KEY"],
                        model_provider="openrouter",
                        base_url="https://openrouter.ai/api/v1",
                        max_tokens=10000, temperature=0.0)



In [5]:
class CustomAgentState(AgentState):
    user_id: str
    preferences: dict

In [6]:

agent = create_agent(
    model=model_advanced,
    tools=[],
    state_schema=CustomAgentState,
    checkpointer=InMemorySaver(),
)


In [7]:

# Custom state can be passed in invoke
result = agent.invoke(
    {
        "messages": [{"role": "user", "content": "Hello"}],
        "user_id": "user_123",
        "preferences": {"theme": "dark"}
    },
    {"configurable": {"thread_id": "1"}})

In [8]:
rprint(result)

{
    'messages': [
        HumanMessage(
            content='Hello',
            additional_kwargs={},
            response_metadata={},
            id='2b79d6ad-aebd-4d58-adbf-77d305f09eb1'
        ),
        AIMessage(
            content='Hello! How can I help you today?',
            additional_kwargs={},
            response_metadata={
                'model_name': 'openai/gpt-5.6-luna-pro',
                'id': 'gen-1789085753-xJLB4GiYCu2RlwNTkHDZ',
                'created': 1789085753,
                'object': 'chat.completion',
                'finish_reason': 'stop',
                'logprobs': None,
                'model_provider': 'openrouter',
                'cost': 0.0003858,
                'cost_details': {
                    'upstream_inference_completions_cost': 8.04e-05,
                    'upstream_inference_prompt_cost': 0.0003054,
                    'upstream_inference_cost': 0.0003858
                }
            },
            id='lc_run--01a08dd2-16c7-7a70-9270-4b4f94033b7f-0',
            tool_calls=[],
            invalid_tool_calls=[],
            usage_metadata={
                'input_tokens': 1527,
                'output_tokens': 67,
                'total_tokens': 1594,
                'input_token_details': {'cache_read': 0, 'cache_creation': 0},
                'output_token_details': {'reasoning': 11}
            }
        )
    ],
    'user_id': 'user_123',
    'preferences': {'theme': 'dark'}
}